In [1]:
!pip install crewai

In [2]:
import os
from google.colab import userdata
from crewai import LLM



# Retrieve the OpenAI API key from Google Colab's user data storage
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Initialize the base LLM model (GPT-4) for agents to use in CrewAI
llm = LLM(model="gpt-4")

In [11]:
from crewai import Agent, Task, Crew
from crewai.tools import tool
import random

# Emotion Analysis Tool (can be improved with real NLP/ML models)
@tool
def analyze_emotion(text: str) -> str:
    """Detects frustration or boredom based on input text."""
    keywords = {"frustrated": ["stuck", "confused", "don't get it"],
                "bored": ["too easy", "boring", "not challenging"]}

    for emotion, words in keywords.items():
        if any(word in text.lower() for word in words):
            return emotion
    return "neutral"

# Tutor Agent
tutor_agent = Agent(
    name="Tutor",
    role="Personalized AI Tutor",
    goal="Help the student understand topics in a way that suits their learning style.",
    backstory="""You are an advanced AI tutor who expalains the given topic to a student. Adapt your explanation style based on student needs after analyzing student's emotion.""",
    tools=[analyze_emotion],
    llm=llm,
    allow_delegation=True,
    verbose=True
)

# Quiz Generator Agent
quiz_agent = Agent(
    name="QuizMaster",
    role="Creates quizzes tailored to student's progress",
    goal="Generate engaging and adaptive quizzes to reinforce learning.",
    backstory="""You are a mentor specialized in making quizzes fun and educational.
    Generate the quiz and their correct answers based on the student's given topic and let the student answer them.
    Once the student answers the question in natural language, pass the quiz questions, correct asnwers and student answers to the feedback_agent for evaluation.""",
    llm=llm,
    verbose=True
)

# Emotion Analyzer Agent
emotion_agent = Agent(
    name="EmotionMonitor",
    role="Monitors student’s emotional state",
    goal="Detects frustration or boredom and suggests teaching adjustments.",
    backstory="An AI that analyzes student emotions and adapts learning accordingly.",
    tools=[analyze_emotion],
    llm=llm,
    verbose=True
)


# Feedback Agent
feedback_agent = Agent(
    name="FeedbackGiver",
    role="Provides personalized feedback",
    goal="Give constructive feedback based on quiz results and emotions.",
    backstory="""You are an expert evaluator who evaluates student performance on the quiz and provides insightful feedback for improvement.
    Tell the student how many questions they have answered correctly in the quiz.""",
    llm=llm,
    verbose=True
)


In [12]:
# Define Tasks
task_teach = Task(
    description="Explain the topic in a way the student understands.",
    agent=tutor_agent,
    human_input=True,
    expected_output="A well-structured explanation of the topic."
)

task_generate_quiz = Task(
    description="Create a short quiz based on what the student has learned.",
    agent=quiz_agent,
    human_input=True,
    expected_output="A short quiz with multiple-choice or open-ended questions."
)

task_analyze_emotion = Task(
    description="Analyze the student's emotional state based on their responses.",
    agent=emotion_agent,
    human_input=True,
    expected_output="An analysis report indicating whether the student is neutral, frustrated, or bored."
)

task_feedback = Task(
    description="Provide feedback on the student's learning progress.",
    agent=feedback_agent,
    expected_output="Constructive feedback on the student's learning journey."
)


In [5]:
# Assemble the Crew
learning_crew = Crew(
    agents=[tutor_agent, quiz_agent, emotion_agent, feedback_agent],
    tasks=[task_teach, task_generate_quiz, task_analyze_emotion, task_feedback],
    verbose=True,
    process_type="sequential"
)

In [13]:
topic = input("Hey, friend! What you wanna learn today?: ")
task_teach.description = f"Explain the topic in a way the student understands. Here's the topic: {topic}"


learning_crew.kickoff()

Hey, friend! What you wanna learn today?: arrays
# Agent: Personalized AI Tutor
## Task: Explain the topic in a way the student understands. Here's the topic: arrays


# Agent: Personalized AI Tutor
## Using tool: analyze_emotion
## Tool Input: 
"{\"text\": \"Please help me understand arrays. I'm struggling to grasp this concept.\"}"
## Tool Output: 
neutral


You ONLY have access to the following tools, and should NEVER make up tools that are not listed here:

Tool Name: analyze_emotion
Tool Arguments: {'text': {'description': None, 'type': 'str'}}
Tool Description: Detects frustration or boredom based on input text.
Tool Name: Delegate work to coworker
Tool Arguments: {'task': {'description': 'The task to delegate', 'type': 'str'}, 'context': {'description': 'The context for the task', 'type': 'str'}, 'coworker': {'description': 'The role/name of the coworker to delegate to', 'type': 'str'}}
Tool Description: Delegate a specific task to one of the following coworkers: Creates quizzes

CrewOutput(raw="Your performance on the quiz shows that you have a good understanding of the core concepts, but there are still areas where you could improve. The questions you struggled with were primarily based on applying concepts in real-life situations. For future quizzes, consider studying not just the concepts but also how they can be used practically. This will not only help you excel in exams but also in your ability to handle real-life problems using what you've learned. On the upside, your neutral emotional state is an asset. It means you are open to receiving feedback and ready to make the necessary changes to improve. Keep up the good spirit! You are on the right track, and with just a bit more effort on practical application of concepts, you will surely excel.", pydantic=None, json_dict=None, tasks_output=[TaskOutput(description="Explain the topic in a way the student understands. Here's the topic: arrays", name=None, expected_output='A well-structured explanation of the 